# Part 1 — Data Preparation & Exploratory Data Analysis
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/01_data_prep_eda.ipynb)

Loads the UCI Appliances Energy Prediction dataset, checks it for missing values, resamples it to hourly resolution, and explores its seasonal structure and stationarity.

In [ ]:
# Colab already ships pandas/numpy/matplotlib/statsmodels/scipy, so this
# is just a safety net if you're running somewhere leaner.
!pip install -q statsmodels

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings('ignore')  # KPSS emits an interpolation warning near its p-value bounds

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'
TARGET = 'Appliances'

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Load and parse the timestamp

In [ ]:
def load_raw_data(path_or_url: str) -> pd.DataFrame:
    """Load the raw 10-minute-resolution dataset and parse the timestamp index."""
    df = pd.read_csv(path_or_url)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    return df

raw = load_raw_data(RAW_CSV_URL)
print(f'Raw shape: {raw.shape}, range: {raw.index.min()} to {raw.index.max()}')
raw.head()

## 2. Check for missing values and timestamp gaps

In [ ]:
def check_missingness(df: pd.DataFrame) -> pd.Series:
    """Report missing values per column and gaps in the expected 10-min time grid.

    UCI's documentation states this dataset has no missing values, but we verify
    rather than assume — and separately check for gaps in the *timestamp*
    sequence itself, since a sensor dropout can skip timestamps entirely without
    showing up as a NaN in any column.
    """
    na_counts = df.isna().sum()
    na_counts = na_counts[na_counts > 0] if na_counts.sum() > 0 else pd.Series({'(none)': 0})

    full_grid = pd.date_range(df.index.min(), df.index.max(), freq='10min')
    missing_timestamps = full_grid.difference(df.index)

    print('Missing values per column (non-zero only):')
    print(na_counts)
    print(f'\nExpected 10-min timestamps: {len(full_grid)}')
    print(f'Actual timestamps present:  {len(df)}')
    print(f'Missing timestamps (gaps in the sequence): {len(missing_timestamps)}')
    return na_counts

_ = check_missingness(raw)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(raw.isna().T, aspect='auto', cmap='viridis', interpolation='none')
ax.set_yticks(range(len(raw.columns)))
ax.set_yticklabels(raw.columns, fontsize=6)
ax.set_xlabel('Time step (10-min resolution)')
ax.set_title('Missing value map — raw dataset (dark = present, yellow = missing)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '01_raw_missingness.png')
plt.show()

## 3. Resample to hourly resolution

In [ ]:
def resample_hourly(df: pd.DataFrame) -> pd.DataFrame:
    """Resample the 10-minute data to hourly resolution.

    Energy variables (Appliances, lights) are metered Wh *per 10-minute
    interval*, so they are summed across each hour to get total Wh consumed
    that hour. Sensor/weather variables are instantaneous readings, so they
    are averaged. The two random variables (rv1, rv2) included by the dataset
    authors for benchmarking are dropped — they carry no predictive information.
    """
    energy_cols = ['Appliances', 'lights']
    sensor_cols = [c for c in df.columns if c not in energy_cols + ['rv1', 'rv2']]

    hourly_energy = df[energy_cols].resample('h').sum()
    hourly_sensors = df[sensor_cols].resample('h').mean()
    return pd.concat([hourly_energy, hourly_sensors], axis=1)

hourly = resample_hourly(raw)
hourly.to_csv(DATA_DIR / 'energydata_hourly.csv')
print(f'Hourly shape: {hourly.shape}, range: {hourly.index.min()} to {hourly.index.max()}')
hourly.head()

## 4. Exploratory plots

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hourly.index, hourly[TARGET], linewidth=0.6, color='#1f5b8a')
ax.set_title('Hourly appliance energy use — full series')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '02_hourly_timeseries.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(hourly[TARGET], bins=50, color='#1f5b8a', edgecolor='white')
axes[0].set_title('Distribution of hourly Appliances (Wh)'); axes[0].set_xlabel('Wh / hour')
axes[1].hist(np.log1p(hourly[TARGET]), bins=50, color='#2e8a5b', edgecolor='white')
axes[1].set_title('Distribution of log(1 + Appliances)'); axes[1].set_xlabel('log(1 + Wh / hour)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '03_distribution.png')
plt.show()

In [ ]:
tmp = hourly.copy()
tmp['hour'] = tmp.index.hour
tmp['dayofweek'] = tmp.index.dayofweek  # 0 = Monday

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hourly_profile = tmp.groupby('hour')[TARGET].mean()
axes[0].plot(hourly_profile.index, hourly_profile.values, marker='o', color='#1f5b8a')
axes[0].set_title('Average appliance use by hour of day'); axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('Mean Wh / hour'); axes[0].set_xticks(range(0, 24, 2))

dow_profile = tmp.groupby('dayofweek')[TARGET].mean()
axes[1].bar(dow_profile.index, dow_profile.values, color='#2e8a5b')
axes[1].set_title('Average appliance use by day of week'); axes[1].set_xlabel('Day of week (0 = Monday)')
axes[1].set_xticks(range(7))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '04_daily_weekly_profile.png')
plt.show()

## 5. STL decomposition
Using a 24-hour period. STL (rather than classical `seasonal_decompose`) is used because appliance energy use is noisy and not perfectly additive — STL's locally-weighted regression handles that more robustly and lets the seasonal component drift slowly across the ~4.5 months of data.

In [ ]:
series = hourly[TARGET].asfreq('h').interpolate()
stl = STL(series, period=24, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(series.index, result.observed, color='#1f5b8a', linewidth=0.6); axes[0].set_ylabel('Observed')
axes[1].plot(series.index, result.trend, color='#c0392b', linewidth=0.8); axes[1].set_ylabel('Trend')
axes[2].plot(series.index, result.seasonal, color='#2e8a5b', linewidth=0.5); axes[2].set_ylabel('Daily seasonal')
axes[3].plot(series.index, result.resid, color='#7f7f7f', linewidth=0.4); axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')
fig.suptitle('STL decomposition (24h period) — hourly Appliances')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '05_stl_decomposition.png')
plt.show()

## 6. Autocorrelation (ACF / PACF)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plot_acf(hourly[TARGET].dropna(), lags=72, ax=axes[0])
axes[0].set_title('ACF — hourly Appliances (up to 72h lag)')
plot_pacf(hourly[TARGET].dropna(), lags=72, ax=axes[1], method='ywm')
axes[1].set_title('PACF — hourly Appliances (up to 72h lag)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / '06_acf_pacf.png')
plt.show()

## 7. Stationarity tests (ADF + KPSS)
Both tests are reported because they have opposite null hypotheses (ADF: H0 = unit root/non-stationary; KPSS: H0 = stationary) — agreement between them is much stronger evidence than either alone.

In [ ]:
def report_stationarity(name: str, s: pd.Series) -> None:
    s = s.dropna()
    adf_stat, adf_p, *_ = adfuller(s, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(s, regression='c', nlags='auto')
    print(f'\n--- {name} ---')
    print(f'ADF statistic:  {adf_stat:.4f}   p-value: {adf_p:.4f}  '
          f"({'stationary (reject H0)' if adf_p < 0.05 else 'non-stationary (fail to reject H0)'})")
    print(f'KPSS statistic: {kpss_stat:.4f}   p-value: {kpss_p:.4f}  '
          f"({'non-stationary (reject H0)' if kpss_p < 0.05 else 'stationary (fail to reject H0)'})")

report_stationarity('Level series', hourly[TARGET])
report_stationarity('First difference', hourly[TARGET].diff())